# Co-scientist Notebook

Run any co-scientist workflow directly from this notebook — no GUI required.

**Prerequisites:**  
- venv activated: `venv\Scripts\activate` (Windows) or `source venv/bin/activate` (Linux/Mac)  
- `.env` file in the repo root with `ANTHROPIC_API_KEY`  
- Jupyter started from within the activated venv: `jupyter notebook` or `jupyter lab`

In [ ]:
from notebook_api import list_workflows, run_cellatria, run_st_agent, run_biomni, run_geo_sra

list_workflows()

---
## CellAtria — Single-cell RNA-seq

Fetches GEO data, converts formats if needed, runs QC → clustering → cell type annotation.

In [ ]:
# Extract metadata from a GEO accession
result = run_cellatria(
    "GSE284775",
    analysis="GEO Dataset Retrieval",
    gene="IFNG",
    project="cellatria_demo",
)
print("Run dir:", result["run_dir"])

In [ ]:
# Full pipeline: download → QC → cluster → annotate
result = run_cellatria(
    "GSE284775",
    analysis="Full Pipeline",
    gene="IFNG",
    project="cellatria_demo",
    model="claude-sonnet-4-6",   # sonnet recommended for full pipeline
)
print("Run dir:", result["run_dir"])

In [ ]:
# Extract metadata from an article URL
result = run_cellatria(
    "https://www.nature.com/articles/s41467-021-27729-z",
    analysis="Metadata Extraction (URL)",
    gene="IFNG",
    project="cellatria_demo",
)
print(result["output"][-2000:])

---
## ST Agent — Spatial Transcriptomics

In [ ]:
H5AD = "data/sample.h5ad"   # replace with your file

result = run_st_agent(
    H5AD,
    gene="IFNG",
    analysis="Spatial Gene Expression",
    project="st_demo",
)
print("Plots saved to:", result["run_dir"])

In [ ]:
# Full analysis report
result = run_st_agent(
    H5AD,
    gene="IFNG",
    analysis="Full Analysis Report",
    project="st_demo",
    model="claude-sonnet-4-6",
)
print(result["output"][-3000:])

---
## Biomni — General Biomedical Agent

In [ ]:
result = run_biomni(
    "Characterize the role of IFNG in tumor immune evasion. "
    "Search recent literature and summarize key findings.",
    project="biomni_demo",
)
print(result["output"][-3000:])

---
## GEO/SRA — Dataset Search and DEG Analysis

In [ ]:
result = run_geo_sra(
    "Download GSE96058 and run DESeq2 differential expression analysis for IFNG.",
    project="geo_demo",
)
print(result["output"][-3000:])

---
## Custom Workflow — Free-form Prompt

Use `run_workflow()` to pass any prompt to any workflow.

In [ ]:
from notebook_api import run_workflow

result = run_workflow(
    "CellAtria",
    "Convert the BD Rhapsody files in data/GSM8693206 to h5ad format, "
    "then run CellExpress with doublet_method=scrublet on human PBMC samples.",
    model="claude-sonnet-4-6",
    project="custom_run",
    timeout=3600,
)
print("Success:", result["success"])
print("Output dir:", result["run_dir"])

---
## Display Results

After any run, inspect results inline.

In [ ]:
import os
from IPython.display import Image, display

run_dir = result["run_dir"]

# Show all PNG plots produced by the run
for png in sorted(run_dir.glob("*.png")):
    print(png.name)
    display(Image(filename=str(png)))

In [ ]:
# Print the final LLM answer
print(result["result"].read_text(encoding="utf-8")[-4000:])